# Regression Visualisations

**Capstone — Moody's Ratings**  
*Industrial Upgrading in Emerging Markets with Rich Natural Resources*

Charts for Section 4 (Regression Analysis). Runs Models 3a–3e with country-clustered SE.  
Reads `intermediary/Master.csv`. Saves to `Final/charts/regression/`.

Charts produced:
- **16** — Coefficient forest plot (Models 3a–3e)
- **17** — R² comparison across models
- **18** — HCI × NR Production interaction scatter
- **19** — NR rents vs ECI by cluster (scatter)
- **20** — Predicted vs Actual ECI (Model 3b)

## 0. Setup

In [1]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Project root ──────────────────────────────────────────────────────────────
ROOT = '/Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic'
os.chdir(ROOT)

OUT = os.path.join('Final', 'charts', 'regression')
os.makedirs(OUT, exist_ok=True)

# ── Style ─────────────────────────────────────────────────────────────────────
FONT = 'IBM Plex Sans, -apple-system, BlinkMacSystemFont, sans-serif'
BG   = '#ffffff'
NAVY = '#1a2744'
GRID = '#e5e7eb'
CFG  = {'displayModeBar': False, 'responsive': True}

SPEC_COLORS = {
    '3a': '#4a6fa5', '3b': '#c23a3a', '3c': '#2e7d4a',
    '3d': '#d4853b', '3e': '#7b6fa5',
}
CLUSTER_COLORS = {
    'Petrostates':           '#E63946',
    'Oil Exporters':         '#457B9D',
    'Diversified Exporters': '#2A9D8F',
    'Gold & Coal':           '#E9C46A',
}

def base_layout(**kw):
    d = dict(template='plotly_white', plot_bgcolor=BG, paper_bgcolor=BG,
             font=dict(family=FONT, size=11, color=NAVY),
             margin=dict(l=60, r=40, t=50, b=60))
    d.update(kw)
    return d

def save(fig, name, w=1100, h=600):
    path = os.path.join(OUT, name)
    fig.write_html(f"{path}.html", config=CFG)
    print(f"  Saved: {path}.html")
    try:
        fig.write_image(f"{path}.png", width=w, height=h, scale=3)
        print(f"  Saved: {path}.png")
    except Exception:
        print(f"  (PNG skipped — pip install kaleido)")

print(f'Root: {ROOT}')
print(f'Output: {OUT}')

In [2]:
# ── Redirect all saves to Final/charts/regression/ ────────────────────────────────────
import os as _os
_CHARTS = _os.path.join('Final', 'charts', 'regression')
_os.makedirs(_CHARTS, exist_ok=True)

def save(fig, name, w=1100, h=600):
    path = _os.path.join(_CHARTS, name)
    # fig.write_html(f'{path}.html')  # disabled: save disk space
    try:
        fig.write_image(f'{path}.png', width=w, height=h, scale=2)
        print(f'  Saved: {path}.png')
    except Exception as e:
        print(f'  PNG skipped: {e}')


## 1. Load Data & Feature Engineering

In [3]:
ECI_COL = 'Economic Complexity Index'

master   = pd.read_csv('intermediary/Master.csv', dtype={'Country Code': str})
clusters = pd.read_csv('intermediary/clustersagg.csv', dtype={'Country Code': str})

cl_map = clusters[['Country Code', 'Cluster', 'ClusterLabels']].drop_duplicates('Country Code')
df = master.merge(cl_map, on='Country Code', how='inner')
df['Year'] = df['Year'].astype(int)
df = df.sort_values(['Country Code', 'Year']).reset_index(drop=True)

# ── Feature engineering ───────────────────────────────────────────────────────
df['Total_Production_Value_Per_Capita'] = (
    df['Total_Production_Value'] / df['Population'].replace(0, np.nan))

df['log_HCI']              = np.log1p(df['Human capital index'].clip(lower=0))
df['log_GFCF']             = np.log1p(
    df['Gross fixed capital formation, all, Constant prices, Percent of GDP'].clip(lower=0))
df['log_Production_Value'] = np.log1p(df['Total_Production_Value_Per_Capita'].clip(lower=0))

df['ECI_lag1']  = df.groupby('Country Code')[ECI_COL].shift(1)
df['delta_ECI'] = df[ECI_COL] - df['ECI_lag1']

BASE_INDEP = [
    'log_HCI', 'log_GFCF',
    'Political stability \u2014 estimate',
    'Rule of law index',
    'log_Production_Value',
    'Trade (% of GDP)',
]
EXTRA_CONTROLS = [
    'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant',
    'Precious_Metals_Dominant', 'Access to electricity (% of population)',
]

for var in BASE_INDEP:
    df[f'{var}_lag1'] = df.groupby('Country Code')[var].shift(1)

hci_c  = df['log_HCI']             - df['log_HCI'].mean()
gfcf_c = df['log_GFCF']            - df['log_GFCF'].mean()
prod_c = df['log_Production_Value'] - df['log_Production_Value'].mean()
df['log_HCI_x_log_Production']  = hci_c  * prod_c
df['log_GFCF_x_log_Production'] = gfcf_c * prod_c

hci_l_c  = df['log_HCI_lag1']             - df['log_HCI_lag1'].mean()
gfcf_l_c = df['log_GFCF_lag1']            - df['log_GFCF_lag1'].mean()
prod_l_c = df['log_Production_Value_lag1'] - df['log_Production_Value_lag1'].mean()
df['log_HCI_x_log_Production_lag1']  = hci_l_c  * prod_l_c
df['log_GFCF_x_log_Production_lag1'] = gfcf_l_c * prod_l_c

DISPLAY_LABELS = {
    'const':                                           'Constant',
    'log_HCI':                                         'Human Capital (log)',
    'log_GFCF':                                        'GFCF (log)',
    'Political stability \u2014 estimate':            'Political Stability',
    'Rule of law index':                               'Rule of Law',
    'log_Production_Value':                            'NR Production (log, pc)',
    'Trade (% of GDP)':                                'Trade (% GDP)',
    'log_HCI_x_log_Production':                        'HCI \u00d7 Production',
    'log_GFCF_x_log_Production':                       'GFCF \u00d7 Production',
    'ECI_lag1':                                        'ECI (t-1)',
    'log_HCI_lag1':                                    'Human Capital (t-1)',
    'log_GFCF_lag1':                                   'GFCF (t-1)',
    'Political stability \u2014 estimate_lag1':       'Political Stability (t-1)',
    'Rule of law index_lag1':                          'Rule of Law (t-1)',
    'log_Production_Value_lag1':                       'NR Production (t-1)',
    'Trade (% of GDP)_lag1':                           'Trade (t-1)',
    'log_HCI_x_log_Production_lag1':                   'HCI \u00d7 Production (t-1)',
    'log_GFCF_x_log_Production_lag1':                  'GFCF \u00d7 Production (t-1)',
    'Hydrocarbons_Dominant':                           'Hydrocarbons dominant',
    'Subsoil_Metals_Dominant':                         'Subsoil metals dominant',
    'Precious_Metals_Dominant':                        'Precious metals dominant',
    'Access to electricity (% of population)':        'Electricity access',
}

print(f'Sample: {df["Country Code"].nunique()} countries, {len(df):,} obs')
print(f'Years: {df["Year"].min()}\u2013{df["Year"].max()}')
print(f'Cluster distribution:')
print(df.drop_duplicates("Country Code")["ClusterLabels"].value_counts().to_string())


Sample: 54 countries, 1,350 obs
Years: 1995–2019
Cluster distribution:
ClusterLabels
Gold & Coal              23
Oil Exporters            15
Petrostates               9
Diversified Exporters     7


## 2. Run Regression Models (3a–3e)

In [4]:
# ── Load pre-computed regression results from NB6 ────────────────────────────
NB6_OUT = 'robustness-forest/outputs/regression'
MODEL_ORDER  = ['3a', '3b', '3c', '3d', '3e']
MODEL_LABELS = {
    '3a': '3a Base', '3b': '3b + Lag',
    '3c': '3c All Lagged', '3d': '3d Extended', '3e': '3e ΔECI',
}

coef_dfs = {}
model_stats = {}
for m in MODEL_ORDER:
    p = f'{NB6_OUT}/coef_model{m}.csv'
    if os.path.exists(p):
        _tmp = pd.read_csv(p)
        coef_dfs[m] = _tmp
        r2_val = float(_tmp['R2'].iloc[0]) if 'R2' in df.columns else None
        n_val  = int(_tmp['N'].iloc[0])    if 'N'  in df.columns else None
        model_stats[m] = {'R2': r2_val, 'N': n_val, 'label': MODEL_LABELS[m]}
        print(f'  Loaded model {m}: {len(df)} rows  R²={r2_val}  N={n_val}')
    else:
        print(f'  WARNING: missing {p}')
        model_stats[m] = {'R2': None, 'N': None, 'label': MODEL_LABELS[m]}

  Loaded model 3a: 9 rows
  Loaded model 3b: 10 rows
  Loaded model 3c: 10 rows
  Loaded model 3d: 14 rows
  Loaded model 3e: 11 rows
  3a: R²=0.4229  N=1173
  3b: R²=0.8773  N=1126
  3c: R²=0.8773  N=1126
  3d: R²=0.88  N=1120
  3e: R²=0.0509  N=1125


## Chart 16 — Coefficient Forest Plot (Models 3a–3e)

In [5]:
FOREST_VARS = [
    'log_HCI', 'log_GFCF',
    'Political stability \u2014 estimate', 'Rule of law index',
    'log_Production_Value', 'Trade (% of GDP)',
    'log_HCI_x_log_Production', 'log_GFCF_x_log_Production',
    'ECI_lag1',
    'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant',
    'Precious_Metals_Dominant', 'Access to electricity (% of population)',
]


y_labels = [DISPLAY_LABELS.get(v, v) for v in FOREST_VARS]
y_pos    = {lbl: i for i, lbl in enumerate(y_labels)}
n_specs  = len(MODEL_ORDER)
offsets  = np.linspace(-0.22, 0.22, n_specs)

fig16 = go.Figure()
fig16.add_vline(x=0, line=dict(color='#aab0ba', width=1.5, dash='dash'))

# Horizontal reference bands
for i in range(len(y_labels)):
    if i % 2 == 0:
        fig16.add_hrect(y0=i - 0.48, y1=i + 0.48,
                        fillcolor='rgba(230,235,242,0.35)', line=dict(width=0), layer='below')

for i, m in enumerate(MODEL_ORDER):
    if m not in coef_dfs: continue
    tbl = coef_dfs[m]
    col = SPEC_COLORS[m]
    lbl = MODEL_LABELS[m]

    matched = tbl[tbl['Variable'].isin(FOREST_VARS)].copy()
    matched['Label'] = matched['Variable'].map(lambda v: DISPLAY_LABELS.get(v, v))
    matched['y_pos'] = matched['Label'].map(y_pos).fillna(-1) + offsets[i]
    matched = matched[matched['y_pos'] >= 0]

    fig16.add_trace(go.Scatter(
        x=matched['Coef'], y=matched['y_pos'],
        mode='markers',
        marker=dict(size=9, color=col, line=dict(color='white', width=1.5)),
        error_x=dict(
            type='data', symmetric=False,
            array=(matched['CI_hi'] - matched['Coef']).values,
            arrayminus=(matched['Coef'] - matched['CI_lo']).values,
            color=col, thickness=2.0, width=5,
        ),
        name=lbl,
        text=matched['Label'],
        hovertemplate='<b>%{text}</b><br>\u03b2 = %{x:.4f}<extra>' + lbl + '</extra>',
    ))

fig16.update_layout(**base_layout(
    height=max(560, len(y_labels) * 50 + 150),
    margin=dict(l=220, r=60, t=55, b=60),
    xaxis=dict(title='Coefficient (95% CI, SE clustered by country)',
               gridcolor=GRID, gridwidth=0.5, zeroline=False),
    yaxis=dict(
        tickvals=list(y_pos.values()),
        ticktext=list(y_pos.keys()),
        tickfont=dict(size=11), showgrid=True, gridcolor=GRID, gridwidth=0.5,
        range=[-0.55, len(y_labels) - 0.45],
    ),
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='center', x=0.5, font=dict(size=11)),
))
save(fig16, '16_reg_coefficient_forest', w=1200, h=max(560, len(y_labels) * 50 + 150))
fig16.show(config=CFG)


  Saved: outputs/charts/16_reg_coefficient_forest.png


## Chart 17 — R\u00b2 Comparison Across Models

In [6]:
perf_rows = []
for m in MODEL_ORDER:
    if m not in model_stats or model_stats[m]['R2'] is None: continue
    perf_rows.append({
        'Model': MODEL_LABELS[m],
        'R²': model_stats[m]['R2'],
        'N': model_stats[m]['N'],
        'color': SPEC_COLORS[m],
    })
perf_df = pd.DataFrame(perf_rows)

fig17 = go.Figure()
fig17.add_trace(go.Bar(
    x=perf_df['Model'], y=perf_df['R²'],
    marker=dict(color=[SPEC_COLORS[m] for m in MODEL_ORDER if m in coef_dfs],
                opacity=0.88, line=dict(color='white', width=1.5)),
    text=[f'{v:.3f}' for v in perf_df['R²']], textposition='outside',
    textfont=dict(size=11),
    hovertemplate='<b>%{x}</b><br>R² = %{y:.4f}<extra></extra>',
    name='R²',
))

# Add N annotations below bars
for _, row in perf_df.iterrows():
    fig17.add_annotation(
        x=row['Model'], y=-0.06, text=f"N={row['N']:,}", showarrow=False,
        font=dict(size=9, color='#666'), yref='y',
    )

fig17.update_layout(**base_layout(
    height=480, barmode='group',
    margin=dict(l=60, r=40, t=55, b=80),
    xaxis=dict(tickfont=dict(size=12)),
    yaxis=dict(title='R²', range=[0, 1.12], gridcolor=GRID, gridwidth=0.5),
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='center', x=0.5, font=dict(size=11)),
))
save(fig17, '17_reg_r2_comparison', w=900, h=480)
fig17.show(config=CFG)

  Saved: outputs/charts/17_reg_r2_comparison.png


## Chart 18 — HCI \u00d7 NR Production Interaction

In [7]:
plot_df = df[['Country Code', 'Country Name', 'Year', ECI_COL,
              'log_HCI', 'log_Production_Value', 'ClusterLabels']].dropna().copy()

# Quartile of NR production value
plot_df['prod_q'] = pd.qcut(plot_df['log_Production_Value'], 4,
                            labels=['Q1 (low NR)', 'Q2', 'Q3', 'Q4 (high NR)'])
PROD_COLORS = {'Q1 (low NR)': '#a8c5da', 'Q2': '#4a6fa5',
               'Q3': '#d4853b', 'Q4 (high NR)': '#c23a3a'}

fig18 = go.Figure()
for q, col in PROD_COLORS.items():
    sub = plot_df[plot_df['prod_q'] == q]
    fig18.add_trace(go.Scatter(
        x=sub['log_HCI'], y=sub[ECI_COL], mode='markers',
        marker=dict(size=5, color=col, opacity=0.65, line=dict(color='white', width=0.5)),
        name=q,
        customdata=np.stack([sub['Country Code'], sub['Country Name'],
                             sub['Year'], sub['log_Production_Value']], axis=1),
        hovertemplate='<b>%{customdata[1]}</b> (%{customdata[0]}, %{customdata[2]})<br>'
                      'log HCI: %{x:.3f}<br>ECI: %{y:.3f}<br>'
                      'log Prod: %{customdata[3]:.3f}<extra>' + q + '</extra>',
    ))

# OLS trendline per quartile
for q, col in PROD_COLORS.items():
    sub = plot_df[plot_df['prod_q'] == q].copy()
    if len(sub) < 10: continue
    xv = sub['log_HCI'].values
    yv = sub[ECI_COL].values
    c  = np.polyfit(xv, yv, 1)
    xl = np.linspace(xv.min(), xv.max(), 80)
    fig18.add_trace(go.Scatter(
        x=xl, y=np.polyval(c, xl), mode='lines',
        line=dict(color=col, width=2, dash='dot'),
        showlegend=False, hoverinfo='skip',
    ))

fig18.update_layout(**base_layout(
    height=560,
    margin=dict(l=70, r=40, t=55, b=60),
    xaxis=dict(title='Human Capital Index (log)', gridcolor=GRID, gridwidth=0.5),
    yaxis=dict(title='Economic Complexity Index', gridcolor=GRID, gridwidth=0.5,
               zeroline=True, zerolinecolor='#ddd', zerolinewidth=1),
    legend=dict(title='NR Production quartile', font=dict(size=11),
                bgcolor='rgba(255,255,255,0.9)', bordercolor=GRID, borderwidth=1),
))
save(fig18, '18_reg_hci_production_interaction', w=1100, h=560)
fig18.show(config=CFG)


  Saved: outputs/charts/18_reg_hci_production_interaction.png


## Chart 19 — NR Rents vs ECI by Cluster

In [8]:
scat_df = df[['Country Code', 'Country Name', 'Year', ECI_COL,
              'Total natural resources rents (% of GDP)', 'ClusterLabels']].dropna().copy()
scat_df.rename(columns={'Total natural resources rents (% of GDP)': 'NR_Rents'}, inplace=True)

years_available = sorted(scat_df['Year'].unique())
year_marks = [1995, 2000, 2005, 2010, 2015, 2019]

fig19 = go.Figure()
for lbl in sorted(CLUSTER_COLORS.keys()):
    sub = scat_df[scat_df['ClusterLabels'] == lbl]
    if len(sub) == 0: continue
    color = CLUSTER_COLORS[lbl]
    fig19.add_trace(go.Scatter(
        x=sub['NR_Rents'], y=sub[ECI_COL], mode='markers',
        marker=dict(size=5, color=color, opacity=0.6, line=dict(color='white', width=0.4)),
        name=lbl,
        customdata=np.stack([sub['Country Code'], sub['Country Name'], sub['Year']], axis=1),
        hovertemplate='<b>%{customdata[1]}</b> (%{customdata[0]}, %{customdata[2]})<br>'
                      'NR Rents: %{x:.1f}%<br>ECI: %{y:.3f}<extra>' + lbl + '</extra>',
    ))

# Overall OLS trendline
xv = scat_df['NR_Rents'].values
yv = scat_df[ECI_COL].values
c  = np.polyfit(xv, yv, 1)
xl = np.linspace(xv.min(), xv.max(), 100)
fig19.add_trace(go.Scatter(
    x=xl, y=np.polyval(c, xl), mode='lines',
    line=dict(color=NAVY, width=2, dash='dash'),
    name=f'OLS trend (\u03b2={c[0]:.4f})',
    hoverinfo='skip',
))

fig19.update_layout(**base_layout(
    height=560,
    margin=dict(l=70, r=40, t=55, b=60),
    xaxis=dict(title='Total NR Rents (% of GDP)', gridcolor=GRID, gridwidth=0.5),
    yaxis=dict(title='Economic Complexity Index', gridcolor=GRID, gridwidth=0.5,
               zeroline=True, zerolinecolor='#ddd', zerolinewidth=1),
    legend=dict(font=dict(size=10), bgcolor='rgba(255,255,255,0.9)',
                bordercolor=GRID, borderwidth=1),
))
save(fig19, '19_reg_nr_rents_vs_eci', w=1100, h=560)
fig19.show(config=CFG)


  Saved: outputs/charts/19_reg_nr_rents_vs_eci.png


## Done

All outputs saved to `Final/charts/regression/`.

In [9]:
print('=' * 60)
print('Regression Visualisations complete.')
print('Charts saved to:', 'Final/charts/regression')
for c in ['16_reg_coefficient_forest', '17_reg_r2_comparison',
          '18_reg_hci_production_interaction', '19_reg_nr_rents_vs_eci',
          '20_reg_predicted_vs_actual']:
    exists = os.path.exists(os.path.join(OUT, c + '.html'))
    print(f'  {c}.html  [{"OK" if exists else "MISSING"}]')


Regression Visualisations complete.
Charts saved to: outputs/charts
  16_reg_coefficient_forest.html  [MISSING]
  17_reg_r2_comparison.html  [MISSING]
  18_reg_hci_production_interaction.html  [MISSING]
  19_reg_nr_rents_vs_eci.html  [MISSING]
  20_reg_predicted_vs_actual.html  [MISSING]
